In [3]:
import pandas as pd

# Absolute path — works no matter where the notebook file is saved
Data_path = r"D:\Python Project\marketpulseAi\data\raw\\"

ecommerce = {
    "customers": pd.read_csv(Data_path + "olist_customers_dataset.csv"),
    "orders": pd.read_csv(Data_path + "olist_orders_dataset.csv"),
    "order_items": pd.read_csv(Data_path + "olist_order_items_dataset.csv"),
    "payments": pd.read_csv(Data_path + "olist_order_payments_dataset.csv"),
    "review": pd.read_csv(Data_path + "olist_order_reviews_dataset.csv"),
    "products": pd.read_csv(Data_path + "olist_products_dataset.csv"),
    "sellers": pd.read_csv(Data_path + "olist_sellers_dataset.csv"),
    "geolocation": pd.read_csv(Data_path + "olist_geolocation_dataset.csv"),
    "category_transalation": pd.read_csv(Data_path + "product_category_name_translation.csv"),
}

# Sanity check: print shape of every table
for name, df in ecommerce.items():
    print(f"{name:25s} → {df.shape[0]:>7,} rows | {df.shape[1]} columns")

customers                 →  99,441 rows | 5 columns
orders                    →  99,441 rows | 8 columns
order_items               → 112,650 rows | 7 columns
payments                  → 103,886 rows | 5 columns
review                    →  99,224 rows | 7 columns
products                  →  32,951 rows | 9 columns
sellers                   →   3,095 rows | 4 columns
geolocation               → 1,000,163 rows | 5 columns
category_transalation     →      71 rows | 2 columns


In [8]:
#Clean And Aggregate Geolocation Table
geo=ecommerce["geolocation"]
print(f"Before dedup: {len(geo):,} rows")
#remove Duplicate row
geo_dedup=geo.drop_duplicates()
lat_ok=geo_dedup["geolocation_lat"].between(-34,5)
lng_ok=geo_dedup["geolocation_lng"].between(-74,-34)
#filter out coordinates outside Brazil's real bounds
geo_valid=geo_dedup[lat_ok & lng_ok]
print(f"After removing out-of-bounds rows: {len(geo_valid):,} rows")
#Aggregate to one row per zip prefix
geo_clean=geo_valid.groupby("geolocation_zip_code_prefix").agg(
geolocation_lat=("geolocation_lat", "mean"),
    geolocation_lng=("geolocation_lng", "mean"),
    geolocation_city=("geolocation_city", lambda x: x.mode()[0]),
    geolocation_state=("geolocation_state", lambda x: x.mode()[0]),
).reset_index()
print(f"\nFinal geo_clean shape: {geo_clean.shape}")
print(f"Duplicate zip prefixes remaining: {geo_clean['geolocation_zip_code_prefix'].duplicated().sum()}")
print(geo_clean.head())
print(f"geo_dedup rows: {len(geo_dedup):,}")
print(f"Difference from original: {len(geo) - len(geo_dedup):,}")    


Before dedup: 1,000,163 rows
After removing out-of-bounds rows: 738,299 rows

Final geo_clean shape: (19010, 5)
Duplicate zip prefixes remaining: 0
   geolocation_zip_code_prefix  geolocation_lat  geolocation_lng  \
0                         1001       -23.550227       -46.634039   
1                         1002       -23.547657       -46.634991   
2                         1003       -23.549000       -46.635582   
3                         1004       -23.549829       -46.634792   
4                         1005       -23.549547       -46.636406   

  geolocation_city geolocation_state  
0        sao paulo                SP  
1        sao paulo                SP  
2        sao paulo                SP  
3        sao paulo                SP  
4        sao paulo                SP  
geo_dedup rows: 738,332
Difference from original: 261,831
